<a href="https://colab.research.google.com/github/dorian-goueytes/M1_SCE_TT_signal_S2/blob/main/Exemple_ERP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction :

Ce notebook à pour objectif d'illustrer l'utilité de l'approche par réponses évoquées en EEG. Il simule une réponse P300 à un stimuli, et vous invite à observer l'effet du bruit et du délai temporel sur nos réponse.

Pour chaque exercice vous pouvez manipuler différents curseurs pour changer des paramètres, puis ré-exécuter la cellule pour observer l'effet de votre manipulation

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

## Bruit et nombre d'essai

En vous basant sur le code qui suit, qu'observez-vous sur l'ERP en fonction du niveau de bruit et du nombre d'essai?

Quelles recommandations formuleriez-vous pour construire une étude EEG basé sur vos observations?

In [ ]:
def simulate_p300(amp=5, latency=0.3, width=0.08, duration=1, fe=1000):
    time = np.linspace(0, duration, int(fe * duration))
    p300 = amp * np.exp(-0.5 * ((time - latency) / width) ** 2)
    return time, p300

# Generate clean P300 signal
time, signal = simulate_p300()

In [ ]:
trials = []

noise_level = 50 # @param {type:"slider", min:0, max:100, step:2}
n_trials_avg1 = 100 # @param {type:"slider", min:10, max:1000, step:10}
n_trials_avg2 = 1000 # @param {type:"slider", min:10, max:10000, step:50}
for i in range(10000):
    noise = noise_level * np.random.normal(size=time.shape)
    trials.append(signal + noise)

trials = np.array(trials)

plt.figure(figsize=(10,6))
plt.plot(time, np.mean(trials[0:n_trials_avg1], axis=0), alpha=0.6, label="Average of "+str(n_trials_avg1)+" Noisy Trials")
plt.plot(time, np.mean(trials[0:n_trials_avg2], axis=0), alpha=0.6, label="Average of "+str(n_trials_avg2)+" Noisy Trials")
plt.plot(time, signal, linewidth=2, label='True P300 Signal')
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (µV)")
plt.title("Effect of Trial Averaging on P300 Signal-to-Noise Ratio")
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()

## Effet du délai (jitter) temporel sur l'ERP

Avant d'exécuter la cellule suivante et d'observer les résultats, quel serait selon vous l'effet d'avoir un léger décalage temporel dans nos essais individuels au niveau de l'ERP (par exemple, au lieu que le pic de notre réponse soit toujours exactement à 300ms, il peut varier entre 250 et 350ms)?

Après avoir exécuté le code suivant, votre réponse était-elle correcte?

Quel est l'effet du jitter temporel, et quel est selon vous le risque si tu jitter est présent lors d'un étude EEG?

In [ ]:
# Base (perfectly aligned) signal
time, aligned_signal = simulate_p300()

trials = []
signals = []
latencies = []

n_trials = 10000
noise_level = 2 # @param {type:"slider", min:0, max:100, step:2}
percent_jittered = 50 # @param {type:"slider", min:0, max:100, step:5}
jitter_sd = 0.04 # @param {type:"slider", min:0, max:0.3, step:0.01}

count = 20
plt.figure(figsize=(10,6))
for i in range(n_trials):

    # Add Gaussian noise (present in all trials)
    noise = noise_level * np.random.normal(size=time.shape)
    # Decide whether this trial is jittered
    if i < n_trials * (percent_jittered / 100):
        jitter = np.random.normal(0, jitter_sd)
        latency = 0.3 + jitter
    else:
        latency = 0.3
    latencies.append(latency)
    # Generate trial-specific P300
    _, signal = simulate_p300(latency=latency)

    signals.append(signal)
    trials.append(signal + noise)
    if i<count:
      plt.plot(time, signal)
plt.axvline(0.3, color = 'k', linestyle = '--')
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (µV)")
plt.title("Example of 20 trials with temporal jitter (pure signal, no noise)")
plt.grid()
plt.tight_layout()
plt.show()

trials = np.array(trials)
signals = np.array(signals)

plt.figure(figsize=(10,6))
np.random.shuffle(trials)
plt.plot(time, np.mean(trials[0:100], axis=0), alpha=0.6, label="Average of 1000 Trials")
plt.plot(time, np.mean(trials[0:1000], axis=0), alpha=0.6, label="Average of 10000 Trials")
plt.plot(time, aligned_signal,  linewidth=2, label="Perfectly Aligned P300")

plt.xlabel("Time (s)")
plt.ylabel("Amplitude (µV)")
plt.title("Effect of Temporal Jitter on ERP Averaging")
plt.axvline(0.3, color = 'k', linestyle = '--')
plt.grid()
plt.legend()
plt.tight_layout()
plt.show()